# Fabric Metadata & Data Quality Assessment Framework

## 1. Overview & Architecture
This framework provides an automated, end-to-end data quality assessment for Microsoft Fabric Lakehouses. It extracts structural metadata, profiles columns, evaluates custom quality rules, generates an aggregate quality index, and outputs an actionable summary report.

### Platform Placement
```
             Microsoft Fabric
                    │
                 OneLake
                    │
                 Lakehouse
                    │
             ┌──────┴──────┐
             │             │
          Metadata      Dataset
             │             │
             └──────┬──────┘
                    │
             Data Profiling
                    │
          Data Quality Rules
                    │
             Quality Score
                    │
             Assessment Report
```

## 2. Configuration & Mode Selection
Set `USE_SYNTHETIC_DATA = True` to run the demonstration mode without external dependencies. Set to `False` and configure `TARGET_TABLE` to profile an existing Delta table in your Fabric Lakehouse.

In [ ]:
# Configuration Parameters
USE_SYNTHETIC_DATA = True
TARGET_TABLE = "default_lakehouse_table"  # Used if USE_SYNTHETIC_DATA is False
SYNTHETIC_ROW_COUNT = 10000

print(f"Execution Mode: {'Synthetic Data Generation' if USE_SYNTHETIC_DATA else f'Lakehouse Table ({TARGET_TABLE})'}")

## 3. Data Ingestion / Synthetic Data Generation
Generates a synthetic e-commerce retail dataset containing artificial data quality anomalies (nulls, duplicates, invalid quantities, negative prices) to demonstrate framework capabilities.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnull, sum as _sum, avg, min as _min, max as _max, lit, rand, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
import datetime

spark = SparkSession.builder.getOrCreate()

if USE_SYNTHETIC_DATA:
    # Generate base dataset with intentional data quality issues
    df_base = spark.range(0, SYNTHETIC_ROW_COUNT).select(
        # Force ~1.4% duplicate order IDs
        when(rand() < 0.014, (col("id") % 100).cast("integer")).otherwise(col("id").cast("integer")).alias("order_id"),
        # 0.2% null customer IDs
        when(rand() < 0.002, lit(None)).otherwise((col("id") % 1000 + 100).cast("string")).alias("customer_id"),
        (col("id") % 500 + 1).cast("integer").alias("product_id"),
        # 0.0% invalid quantities (all >= 1)
        (expr("abs(cast(rand() * 5 as int)) + 1")).alias("quantity"),
        # 0.3% invalid sales amounts (negative prices)
        when(rand() < 0.003, -10.0).otherwise(expr("round(rand() * 100 + 10, 2)")).alias("unit_price"),
        # Timestamp
        expr("current_timestamp()").alias("transaction_timestamp")
    )
    df_target = df_base
    table_name = "demo_synthetic_retail"
else:
    df_target = spark.table(TARGET_TABLE)
    table_name = TARGET_TABLE

df_target.cache()
total_rows = df_target.count()
print(f"Target Dataset '{table_name}' loaded successfully. Total Rows: {total_rows:,}")

## 4. Metadata Discovery
Extracts schema specifications, column physical data types, nullability properties, and table-level dimensions.

In [ ]:
metadata_schema = []
for field in df_target.schema.fields:
    metadata_schema.append({
        "Column Name": field.name,
        "Data Type": field.dataType.simpleString(),
        "Nullable": field.nullable
    })

df_metadata = spark.createDataFrame(metadata_schema)
print(f"=== METADATA PROFILE: {table_name} ===")
df_metadata.show(truncate=False)

## 5. Statistical Data Profiling
Computes statistical profiles for every attribute, including null ratios, distinct cardinality, min/max values, and duplicate frequency.

In [ ]:
from pyspark.sql.functions import countDistinct

profile_exprs = []
for c in df_target.columns:
    profile_exprs.extend([
        count(when(col(c).isNull(), 1)).alias(f"{c}__null_cnt"),
        countDistinct(c).alias(f"{c}__distinct_cnt"),
        _min(c).cast("string").alias(f"{c}__min_val"),
        _max(c).cast("string").alias(f"{c}__max_val")
    ])

# Single Spark action execution across all attributes
profile_row = df_target.select(profile_exprs).collect()[0]

profiling_results = []
for c in df_target.columns:
    null_cnt = profile_row[f"{c}__null_cnt"]
    profiling_results.append({
        "Column": c,
        "Null Count": null_cnt,
        "Null Percentage": round((null_cnt / total_rows) * 100, 2),
        "Distinct Values": profile_row[f"{c}__distinct_cnt"],
        "Min Value": str(profile_row[f"{c}__min_val"]),
        "Max Value": str(profile_row[f"{c}__max_val"])
    })

df_profile = spark.createDataFrame(profiling_results)
print("=== DATA PROFILING SUMMARY ===")
df_profile.show(truncate=False)

## 6. Data Quality Rules Execution
Applies logical data quality validations (Completeness, Uniqueness, Validity) across target columns.

In [ ]:
dq_checks = []

# Check 1: Completeness - Customer ID Nulls
null_cust_cnt = df_target.filter(col("customer_id").isNull()).count()
null_cust_pct = round((null_cust_cnt / total_rows) * 100, 2)
dq_checks.append({
    "CheckName": "Null customer IDs",
    "Dimension": "Completeness",
    "MetricValue": f"{null_cust_pct}%",
    "Status": "PASS" if null_cust_pct <= 0.5 else "FAIL",
    "Weight": 20,
    "Passed": 1 if null_cust_pct <= 0.5 else 0
})

# Check 2: Uniqueness - Order ID Duplicates
distinct_orders = df_target.select("order_id").distinct().count()
dup_order_pct = round(((total_rows - distinct_orders) / total_rows) * 100, 2)
dq_checks.append({
    "CheckName": "Duplicate orders",
    "Dimension": "Uniqueness",
    "MetricValue": f"{dup_order_pct}%",
    "Status": "WARNING" if 0.5 < dup_order_pct <= 2.0 else ("PASS" if dup_order_pct <= 0.5 else "FAIL"),
    "Weight": 25,
    "Passed": 0.75 if 0.5 < dup_order_pct <= 2.0 else (1 if dup_order_pct <= 0.5 else 0)
})

# Check 3: Validity - Quantity > 0
invalid_qty_cnt = df_target.filter(col("quantity") <= 0).count()
invalid_qty_pct = round((invalid_qty_cnt / total_rows) * 100, 2)
dq_checks.append({
    "CheckName": "Invalid quantities",
    "Dimension": "Validity",
    "MetricValue": f"{invalid_qty_pct}%",
    "Status": "PASS" if invalid_qty_pct == 0.0 else "FAIL",
    "Weight": 25,
    "Passed": 1 if invalid_qty_pct == 0.0 else 0
})

# Check 4: Validity - Unit Price > 0
invalid_price_cnt = df_target.filter(col("unit_price") <= 0).count()
invalid_price_pct = round((invalid_price_cnt / total_rows) * 100, 2)
dq_checks.append({
    "CheckName": "Invalid sales amount",
    "Dimension": "Validity",
    "MetricValue": f"{invalid_price_pct}%",
    "Status": "PASS" if invalid_price_pct <= 0.5 else "FAIL",
    "Weight": 30,
    "Passed": 1 if invalid_price_pct <= 0.5 else 0
})

df_dq_results = spark.createDataFrame(dq_checks)
df_dq_results.select("CheckName", "Dimension", "MetricValue", "Status").show(truncate=False)

## 7. Data Quality Index (DQI) Calculation
Computes a weighted total percentage index evaluating overall dataset health.

In [ ]:
total_weight = sum([r["Weight"] for r in dq_checks])
weighted_passed = sum([r["Weight"] * r["Passed"] for r in dq_checks])
overall_dq_score = round((weighted_passed / total_weight) * 100, 1)

print("=" * 45)
print(f"  OVERALL DATA QUALITY SCORE: {overall_dq_score}%")
print("=" * 45)

## 8. Final Assessment Report
Formats an executive scorecard table showing individual validation metrics.

In [ ]:
print("+-----------------------+--------------+--------+")
print("| CheckResult           | Status       | Metric |")
print("+-----------------------+--------------+--------+")
print(f"| Row count             | PASS         | {total_rows:,}")
for r in dq_checks:
    check = r['CheckName'].ljust(21)
    status = r['Status'].ljust(12)
    metric = r['MetricValue'].ljust(6)
    print(f"| {check} | {status} | {metric} |")
print("+-----------------------+--------------+--------+")

## 9. Automated Recommendations & Remediation
Provides automated next steps based on the validation rule output.

In [ ]:
print("=== AUTOMATED REMEDIATION RECOMMENDATIONS ===")
for r in dq_checks:
    if r["Status"] == "WARNING":
        print(f"- [WARNING] {r['CheckName']}: Metric is {r['MetricValue']}. Apply `dropDuplicates(['order_id'])` transformation downstream.")
    elif r["Status"] == "FAIL":
        print(f"- [ACTION REQUIRED] {r['CheckName']}: Metric is {r['MetricValue']}. Investigate source pipeline for bad input data.")

if overall_dq_score >= 85.0:
    print("\nDataset Status: APPROVED for Gold Layer / Downstream Analytics Consumption.")
else:
    print("\nDataset Status: REJECTED for Production Use. Data remediation required.")